In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import urllib

load_dotenv()

In [ ]:
# 상수 선언
AUTH_KEY = os.getenv('CLIMATE_API_KEY')
BASE_URL = 'https://apihub.kma.go.kr/api/typ02/openApi/SfcMtlyInfoService'

In [ ]:
def request_kma(service, page_no, num_of_rows, year, month):
    """
    기상청 API 호출 함수

    Params
        service: getMmSumry2(월요약자료2), getMnSumry2, getNote
        page_no: 페이지 번호
        num_of_rows: 한 페이지 결과 수
        year: 년도
        month: 월

    Returns
        str: API 응답 JSON 문자열 (에러 시에도 응답 본문 반환)
    """
    url = f'{BASE_URL}/{service}'
    query_params = {
        'pageNo': page_no,
        'numOfRows': num_of_rows,
        'dataType': 'JSON',
        'year': year,
        'month': month,
        'authKey': AUTH_KEY,
    }
    full_url = f'{url}?{urllib.parse.urlencode(query_params)}'

    try:
        with urllib.request.urlopen(full_url, timeout=30) as response:
            return response.read().decode('utf-8')
    except urllib.error.HTTPError as e:
        return e.read().decode('utf-8')
    except (urllib.error.URLError, OSError, TimeoutError) as e:
        return json.dumps({
            'response': {
                'header': {'resultCode': '98', 'resultMsg': f'네트워크 오류: {e}'}
            }
        })


In [ ]:
def parse_response(response_text):
    """
    API 응답 파싱. resultCode 00인 경우에만 items 반환.
    에러 시 ValueError 발생.
    """
    try:
        data = json.loads(response_text)
    except json.JSONDecodeError as e:
        raise ValueError(f"JSON 파싱 실패: {e}") from e

    header = (data.get('response') or {}).get('header') or {}
    result_code = header.get('resultCode', '00')
    result_msg = header.get('resultMsg', '')

    if result_code != '00':
        raise ValueError(f"[ERROR] API 오류 (code={result_code}): {result_msg}")

    return data['response']['body']['items']['item']

In [ ]:
# 단건 테스트 (api 연결 및 응답 구조 확인용)
response = request_kma('getMmSumry2', page_no='1', num_of_rows='10', year='2016', month='06')

# JSON 응답 파싱
import json
data = json.loads(response)

# 응답 구조 확인
print("=== API 응답 구조 ===")
print(f"Result Code: {data['response']['header']['resultCode']}")
print(f"Result Msg: {data['response']['header']['resultMsg']}")
print(f"Total Count: {data['response']['body']['totalCount']}")

# 첫 번째 관측소 데이터 샘플
if 'item' in data['response']['body']['items']:
    item = data['response']['body']['items']['item'][0]
    if 'month' in item and 'info' in item['month']:
        info_list = item['month']['info']
        print(f"\n=== 관측소 데이터 개수 ===")
        print(f"관측소 수: {len(info_list)}")
        
        # 첫 3개 관측소 샘플
        print(f"\n=== 첫 3개 관측소 데이터 샘플 ===")
        for i, station in enumerate(info_list[:3]):
            print(f"\n[{i+1}] 관측소: {station.get('stn_ko', 'N/A')} (ID: {station.get('stn_id', 'N/A')})")
            print(f"    강수일수: {station.get('rn_day', 'N/A')} | 평균기온차: {station.get('rn', 'N/A')}")
            print(f"    최대풍속: {station.get('ws_max', 'N/A')} | 풍향: {station.get('wd_max', 'N/A')}")
    else:
        print("응답 구조가 예상과 다릅니다.")
        print(f"item 구조: {item}")
else:
    print("items.item이 없습니다.")
    print(f"응답 구조: {data}")

In [ ]:
import time

# ---------------------------------------------------------------------------
# getMmSumry2 데이터 컬럼 (API 문서 기준)
# ---------------------------------------------------------------------------
DATA_COLUMNS = [
    'stn_id', 'stn_ko', 'rn_day', 'rn', 'max_rn_day', 'tm_rn_day',
    'rn_day_cnt1', 'rn_day_cnt2', 'rn_day_cnt3', 'rn_day_cnt4',
    'ev_s', 'ws', 'ws_max', 'wd_max', 'tm_max',
    'cnt1', 'cnt2', 'cnt3', 'cnt4', 'cnt5', 'cnt6', 'cnt7', 'cnt8', 'cnt9',
]


def _parse_api_response(response_text):
    """
    API 응답 JSON 파싱. 에러 시 (파싱 실패, 구조 오류) 예외 발생.
    Returns: (result_code, station_data_list or None, total_count)
    
    API 구조: response.body.items.item[].month.info[] 형태
    - item[].month.info[] 안에 각 관측소별 데이터가 있음
    """
    try:
        data = json.loads(response_text)
    except json.JSONDecodeError as e:
        raise ValueError(f"JSON 파싱 실패: {e}") from e

    response = data.get('response') or {}
    header = response.get('header') or {}
    result_code = str(header.get('resultCode', '00'))

    if result_code == '99':
        return result_code, None, 0

    body = response.get('body') or {}
    total_count = body.get('totalCount', 0)
    items_obj = body.get('items') or {}
    item_raw = items_obj.get('item')

    if item_raw is None:
        return result_code, [], total_count

    # item을 리스트로 변환
    if isinstance(item_raw, list):
        item_list = item_raw
    else:
        item_list = [item_raw]

    # item[].month.info[] 안의 실제 관측소 데이터 추출
    station_data_list = []
    for item in item_list:
        if isinstance(item, dict):
            month_data = item.get('month', {})
            info_list = month_data.get('info', [])
            if isinstance(info_list, list):
                station_data_list.extend(info_list)
            elif info_list:  # info가 단일 객체인 경우
                station_data_list.append(info_list)

    return result_code, station_data_list, total_count


def _create_null_row(year, month):
    """resultCode 99용 행: year, month만 채우고 나머지는 None."""
    row = {'year': year, 'month': month}
    for col in DATA_COLUMNS:
        row[col] = None
    return row


def _create_data_row(year, month, station_data):
    """정상 응답 station_data → 행 변환."""
    row = {'year': year, 'month': month}
    for col in DATA_COLUMNS:
        value = station_data.get(col) if isinstance(station_data, dict) else None
        # "null" 문자열을 None으로 변환
        if value == "null":
            value = None
        row[col] = value
    return row


def fetch_and_collect(start_year=2000, end_year=2025, output_path='weather_mm_sumry2.csv', delay_sec=0.3, rows_per_page=1000):
    """
    2000-01 ~ 2025-12 기간 API 호출 후 CSV 저장 (페이지네이션 지원)
    
    - resultCode 99: 날짜만 유지, 나머지는 null
    - 네트워크/파싱 오류: 해당 연월은 건너뛰고 로그 출력, 계속 진행
    - 페이지네이션: totalCount를 확인하여 모든 페이지 데이터 수집
    """
    rows = []
    error_99_count = 0
    error_count = 0
    total_api_calls = 0

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            page_no = 1
            month_data_collected = False

            while True:
                try:
                    total_api_calls += 1
                    response = request_kma(
                        'getMmSumry2',
                        page_no=str(page_no),
                        num_of_rows=str(rows_per_page),
                        year=str(year),
                        month=f'{month:02d}',
                    )
                    result_code, station_list, total_count = _parse_api_response(response)

                    if result_code == '99':
                        if page_no == 1:
                            rows.append(_create_null_row(year, month))
                            error_99_count += 1
                        break

                    if not station_list:
                        if page_no == 1:
                            print(f"[정보] {year}-{month:02d} 데이터 없음")
                        break

                    # 관측소 데이터 추가
                    for station_data in station_list:
                        rows.append(_create_data_row(year, month, station_data))
                        month_data_collected = True

                    # 페이지네이션 확인
                    if page_no * rows_per_page >= total_count:
                        break

                    page_no += 1
                    time.sleep(delay_sec)

                except (ValueError, KeyError, TypeError) as e:
                    error_count += 1
                    print(f"[경고] {year}-{month:02d} (page {page_no}) 파싱 오류: {e}")
                    if page_no == 1:
                        rows.append(_create_null_row(year, month))
                    break
                except Exception as e:
                    error_count += 1
                    print(f"[경고] {year}-{month:02d} (page {page_no}) 예상치 못한 오류: {e}")
                    if page_no == 1:
                        rows.append(_create_null_row(year, month))
                    break

            # 마지막 페이지 후 대기
            if month_data_collected:
                time.sleep(delay_sec)

    df = pd.DataFrame(rows)

    try:
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
    except (OSError, PermissionError) as e:
        print(f"[오류] CSV 저장 실패: {e}")
        raise

    print(f"\n=== 수집 완료 ===")
    print(f"총 API 호출 수: {total_api_calls}")
    print(f"resultCode 99(발간되지 않은 기간) 건수: {error_99_count}")
    if error_count > 0:
        print(f"파싱/기타 오류 건수: {error_count}")
    print(f"총 행 수: {len(df)}, CSV 저장: {output_path}")
    
    # 연도/월별 데이터 수 확인
    if len(df) > 0 and 'year' in df.columns:
        print(f"\n=== 연도별 데이터 수 ===")
        year_counts = df.groupby('year').size()
        print(year_counts)

    return df, error_99_count

In [ ]:
OUTPUT_PATH = Path('data/weather_mm_sumry2.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

try:
    df, error_99_count = fetch_and_collect(
        start_year=2016,
        end_year=2026,
        output_path=str(OUTPUT_PATH),
        delay_sec=0.3,
    )
    print(f"\n총 데이터 행 수: {len(df)}")
    print(f"결측 기간(99) 수: {error_99_count}")
    print("\n=== 데이터 샘플 (처음 10개) ===")
    print(df.head(10))
    print("\n=== 관측소별 데이터 수 ===")
    if 'stn_ko' in df.columns:
        print(df['stn_ko'].value_counts().head(10))
except Exception as e:
    print(f"[오류] 실행 실패: {e}")
    raise